# Exp8 test inference

Run the current ResNet50 best checkpoint on a Kaggle test-image dataset.

Fixed validation settings (`mAP@0.5 = 0.840254`): input `640`, confidence `0.0001`, class-aware NMS IoU `0.50`, and maximum `300` detections per image. TTA is disabled.

In [ ]:
# Configuration
GITHUB_REPO_URL = "https://github.com/minh071289/YOLOv11-pt.git"
REPO_DIR = "/kaggle/working/YOLOv11-pt"

# Leave these as None to auto-discover them under /kaggle/input.
# Example explicit paths:
# CHECKPOINT_PATH = "/kaggle/input/my-exp8-weights/best.pth"
# TEST_IMAGE_DIR = "/kaggle/input/my-test-dataset/test/images"
CHECKPOINT_PATH = None
TEST_IMAGE_DIR = None

INPUT_SIZE = 640
CONFIDENCE = 0.0001
NMS_IOU = 0.50
MAX_DETECTIONS = 300
BATCH_SIZE = 8

OUTPUT_JSON = "/kaggle/working/test_predictions.json"
OUTPUT_SUBMISSION = "/kaggle/working/submission.csv"
OUTPUT_SUMMARY = "/kaggle/working/inference_summary.json"

In [ ]:
# Clone the complete repository so all root scripts such as predict.py are available.
import os
import shutil
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    "git", "clone", GITHUB_REPO_URL, str(repo_dir),
], check=True)

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
print(f"Repository ready: {repo_dir}")

In [ ]:
# Locate the checkpoint and test images from the two attached Kaggle datasets.
from collections import Counter

IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
KAGGLE_INPUT = Path("/kaggle/input")

def discover_checkpoint(root):
    candidates = list(root.rglob("*.pth"))
    if not candidates:
        raise FileNotFoundError("No .pth checkpoint found under /kaggle/input")
    candidates.sort(
        key=lambda path: (
            path.name.lower() == "best.pth",
            "best" in path.name.lower(),
            "exp8" in str(path).lower(),
        ),
        reverse=True,
    )
    return candidates[0]

def discover_image_dir(root):
    image_files = [
        path for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]
    if not image_files:
        raise FileNotFoundError("No test images found under /kaggle/input")
    counts = Counter(path.parent for path in image_files)
    return counts.most_common(1)[0][0]

checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else discover_checkpoint(KAGGLE_INPUT)
test_image_dir = Path(TEST_IMAGE_DIR) if TEST_IMAGE_DIR else discover_image_dir(KAGGLE_INPUT)

if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Checkpoint does not exist: {checkpoint_path}")
if not test_image_dir.is_dir():
    raise NotADirectoryError(f"Test image directory does not exist: {test_image_dir}")

image_paths = sorted(
    [path for path in test_image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS],
    key=lambda path: path.name,
)
if not image_paths:
    raise RuntimeError(f"No images directly inside: {test_image_dir}")

print(f"Checkpoint: {checkpoint_path}")
print(f"Test images: {test_image_dir}")
print(f"Number of images: {len(image_paths)}")

In [ ]:
# Load the exact architecture saved in the checkpoint.
import cv2
import torch
from tqdm.auto import tqdm

from predict import load_model
from utils import util
from utils.json_dataset import letterbox, scale_boxes_to_original

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle, select a GPU accelerator first.")

device = torch.device("cuda")
model, classes, checkpoint_input_size = load_model(checkpoint_path, device)
if checkpoint_input_size != INPUT_SIZE:
    print(f"Checkpoint input_size={checkpoint_input_size}; using validated INPUT_SIZE={INPUT_SIZE}")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Classes: {classes}")
print(f"Parameters: input={INPUT_SIZE}, conf={CONFIDENCE}, nms={NMS_IOU}, max_det={MAX_DETECTIONS}")

In [ ]:
# Batched CUDA inference. NMS is run per image to avoid the batch time limit in util.non_max_suppression.
import csv
import json
import time
from pathlib import Path

def load_batch(paths):
    tensors = []
    metadata = []
    for image_path in paths:
        image = cv2.imread(str(image_path))
        if image is None:
            raise FileNotFoundError(f"Unable to read image: {image_path}")
        height, width = image.shape[:2]
        resized, ratio, pad = letterbox(image, INPUT_SIZE)
        tensor = torch.from_numpy(resized.transpose((2, 0, 1))[::-1].copy())
        tensors.append(tensor)
        metadata.append((image_path.name, ratio, pad, width, height))
    samples = torch.stack(tensors).to(device, non_blocking=True).float() / 255.0
    return samples, metadata

def format_prediction(detections, metadata):
    image_id, ratio, pad, width, height = metadata
    boxes = []
    if detections.shape[0]:
        detections = detections.detach().cpu()
        detections[:, :4] = scale_boxes_to_original(
            detections[:, :4], ratio, pad, width, height
        )
        for x1, y1, x2, y2, score, class_id in detections.tolist():
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append({
                "class": classes[int(class_id)],
                "confidence": round(float(score), 6),
                "bbox": [
                    round(float(x1), 3), round(float(y1), 3),
                    round(float(x2), 3), round(float(y2), 3),
                ],
            })
    boxes.sort(key=lambda item: item["confidence"], reverse=True)
    if len(boxes) > MAX_DETECTIONS:
        raise AssertionError(f"{image_id} exceeded MAX_DETECTIONS")
    return {"image_id": image_id, "boxes": boxes}

predictions = []
model_seconds = 0.0
wall_start = time.perf_counter()

with torch.inference_mode():
    for start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Detecting"):
        batch_paths = image_paths[start:start + BATCH_SIZE]
        samples, metadata = load_batch(batch_paths)
        torch.cuda.synchronize()
        infer_start = time.perf_counter()
        with torch.amp.autocast(device_type="cuda", enabled=True):
            outputs = model(samples)
        torch.cuda.synchronize()
        model_seconds += time.perf_counter() - infer_start
        outputs = outputs.float()

        for index, image_metadata in enumerate(metadata):
            detections = util.non_max_suppression(
                outputs[index:index + 1],
                confidence_threshold=CONFIDENCE,
                iou_threshold=NMS_IOU,
                max_detections=MAX_DETECTIONS,
            )[0]
            predictions.append(format_prediction(detections, image_metadata))

wall_seconds = time.perf_counter() - wall_start
if len(predictions) != len(image_paths):
    raise AssertionError("Prediction count does not match image count")

Path(OUTPUT_JSON).write_text(
    json.dumps(predictions, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

# Competition submission format: exactly one row per image.
with Path(OUTPUT_SUBMISSION).open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["image_id", "bounding_boxes"])
    writer.writeheader()
    for prediction in predictions:
        submission_boxes = []
        for box in prediction["boxes"]:
            x1, y1, x2, y2 = box["bbox"]
            submission_boxes.append({
                "x_min": x1,
                "y_min": y1,
                "x_max": x2,
                "y_max": y2,
                "class": box["class"],
                "confidence": box["confidence"],
            })
        writer.writerow({
            "image_id": prediction["image_id"],
            "bounding_boxes": json.dumps(submission_boxes, ensure_ascii=False),
        })

summary = {
    "checkpoint": str(checkpoint_path),
    "test_image_dir": str(test_image_dir),
    "num_images": len(image_paths),
    "num_predictions": sum(len(item["boxes"]) for item in predictions),
    "classes": classes,
    "input_size": INPUT_SIZE,
    "confidence": CONFIDENCE,
    "nms_iou": NMS_IOU,
    "max_detections_per_image": MAX_DETECTIONS,
    "tta": False,
    "amp": True,
    "wall_seconds": round(wall_seconds, 3),
    "wall_ms_per_image": round(1000 * wall_seconds / len(image_paths), 3),
    "model_seconds": round(model_seconds, 3),
    "model_ms_per_image": round(1000 * model_seconds / len(image_paths), 3),
}
Path(OUTPUT_SUMMARY).write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print(f"Saved: {OUTPUT_JSON}")
print(f"Saved submission: {OUTPUT_SUBMISSION}")
print(f"Saved: {OUTPUT_SUMMARY}")

In [ ]:
# Optional visual check. This does not alter the saved predictions.
import matplotlib.pyplot as plt

CLASS_COLORS = {
    "person": (255, 56, 56),
    "car": (56, 255, 56),
    "dog": (56, 56, 255),
    "cat": (255, 255, 56),
    "chair": (255, 56, 255),
}

preview_count = min(4, len(predictions))
figure, axes = plt.subplots(preview_count, 1, figsize=(14, 8 * preview_count))
if preview_count == 1:
    axes = [axes]

for axis, prediction in zip(axes, predictions[:preview_count]):
    image = cv2.imread(str(test_image_dir / prediction["image_id"]))
    # Draw only the top 30 boxes in the preview to keep it readable.
    for box in prediction["boxes"][:30]:
        x1, y1, x2, y2 = map(int, box["bbox"])
        color = CLASS_COLORS.get(box["class"], (255, 255, 255))
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        label = f'{box["class"]} {box["confidence"]:.3f}'
        cv2.putText(image, label, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(prediction["image_id"])
    axis.axis("off")

plt.tight_layout()
plt.show()